# OmniShift — Multiply-Free CNN Framework

**Framework**: Apply coordinated PoT quantization to any CNN backbone → fully multiply-free inference.

Four independently toggleable components:
| # | Component | Effect |
|---|-----------|--------|
| 1 | **Sparse Shift** | W ∈ {0, ±2^p} — replaces conv multiplication with bit-shifts + skip-zero |
| 2 | **PoT-BN** | γ/σ → ±2^q — replaces BN scale multiplication with shift |
| 3 | **PoT-Act** | post-ReLU → {0} ∪ {2^p} — activation quantization |
| 4 | **EWGS** | Element-Wise Gradient Scaling — smoother training near quantization boundaries |

Supported backbones: `resnet20`, `resnet32`, `resnet56`, `resnet110`, `vgg11`  
Supported datasets: `cifar10`, `cifar100`, `svhn`, `stl10`, `tiny_imagenet`

## Setup

In [ ]:
import os, subprocess, sys

REPO = '/kaggle/working/OmniShift'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/CryAndRRich/OmniShift.git', REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
!pip install -q pyyaml
print('Setup done. CWD:', os.getcwd())

In [ ]:
import torch
device   = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_name = torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU only'
print(f'Device : {gpu_name}')
print(f'PyTorch: {torch.__version__}')

---
## ▶ Single Run

Edit the config cell below, then run the training cell.

In [ ]:
# ============================================================
# CONFIG — change these to control the experiment
# ============================================================

BACKBONE  = 'resnet20'    # resnet20 | resnet32 | resnet56 | resnet110 | vgg11
DATASET   = 'cifar10'     # cifar10 | cifar100 | svhn | stl10 | tiny_imagenet
RUN_NAME  = 'omnishift'   # prefix for checkpoint + log filenames
SEED      = 42
EPOCHS    = 200

# ── Quantization toggles ─────────────────────────────────────
USE_SPARSE     = True
SPARSE_MODE    = 'learnable'   # 'fixed' | 'learnable'
SPARSITY_RATIO = 0.5           # only used when SPARSE_MODE = 'fixed'

USE_POT_BN  = True
BN_WARMUP   = 30               # epoch at which PoT-BN and PoT-Act activate

USE_POT_ACT    = True
ACT_LEVELS     = 8
ACT_ALPHA_INIT = 4.0

USE_EWGS     = True
EWGS_LAMBDA  = 0.02
# ============================================================

# Build config dict (equivalent to YAML)
cfg = {
    'experiment': {
        'backbone': BACKBONE,
        'dataset':  DATASET,
        'name':     RUN_NAME,
        'seed':     SEED,
    },
    'quantize': {
        'use_sparse':     USE_SPARSE,
        'sparse_mode':    SPARSE_MODE,
        'sparsity_ratio': SPARSITY_RATIO,
        'use_pot_bn':     USE_POT_BN,
        'bn_warmup':      BN_WARMUP,
        'use_pot_act':    USE_POT_ACT,
        'act_levels':     ACT_LEVELS,
        'act_alpha_init': ACT_ALPHA_INIT,
        'use_ewgs':       USE_EWGS,
        'ewgs_lambda':    EWGS_LAMBDA,
    },
    'training': {
        'epochs':          EPOCHS,
        'batch_size':      256,
        'lr':              0.1,
        'momentum':        0.9,
        'weight_decay':    5e-4,
        'warmup_epochs':   0,
        'clip_grad':       1.0,
        'sparsity_lambda': 1e-4,
    },
    'output': {
        'checkpoint_dir': 'checkpoints',
        'log_dir':        'logs',
    },
}
print('Config ready. Backbone:', BACKBONE, '| Dataset:', DATASET)
print('Quantize: sparse={} pot_bn={} pot_act={} ewgs={}'.format(
    USE_SPARSE, USE_POT_BN, USE_POT_ACT, USE_EWGS))

In [ ]:
# Sanity check — quick forward pass before training
from src.models.resnet_cifar import build_model
from src.utils.ops_counter import count_mul_add_shift, count_params
from src.quantize.pot_bn import set_bn_epoch

_m = build_model(BACKBONE, cfg['quantize'], num_classes=10, in_channels=3)
set_bn_epoch(_m, 999)
_x = torch.randn(2, 3, 32, 32)
_out = _m(_x)
_ops = count_mul_add_shift(_m, (1, 3, 32, 32))
print(f'Forward OK. Output: {_out.shape}')
print(f'Params : {count_params(_m)/1e6:.3f}M')
print(f'Energy : {_ops["energy_GpJ"]:.4f} GpJ (post-warmup estimate)')
print(f'Sparsity: {_ops["sparsity"]:.1%}')
del _m, _x, _out, _ops

In [ ]:
# Run training
from scripts.run_experiment import run
result = run(cfg)
print(f"\nFinal: test_acc={result['test_acc']:.4f} | "
      f"energy={result['final_ops']['energy_GpJ']:.4f} GpJ | "
      f"sparsity={result['final_sparsity']:.2%}")

---
## ▶ Ablation Suite

Run multiple configurations by toggling the 4 components.
Edit `ABLATION_CONFIGS` below, then run the cell.

In [ ]:
# ============================================================
# ABLATION CONFIG — define configurations to compare
# Each entry: (name, use_sparse, use_pot_bn, use_pot_act, use_ewgs)
# ============================================================
ABLATION_BACKBONE = 'resnet20'
ABLATION_DATASET  = 'cifar10'
ABLATION_EPOCHS   = 200

ABLATION_CONFIGS = [
    # name                    sparse  pot_bn  pot_act  ewgs
    ('baseline_fp32',         False,  False,  False,   False),
    ('sparse_only',           True,   False,  False,   False),
    ('sparse_potbn',          True,   True,   False,   False),
    ('sparse_potbn_ewgs',     True,   True,   False,   True),
    ('sparse_potbn_potact',   True,   True,   True,    False),
    ('omnishift_full',        True,   True,   True,    True),   # all 4 ON
]
# ============================================================

import copy

ablation_results = []

for (run_name, use_sparse, use_pot_bn, use_pot_act, use_ewgs) in ABLATION_CONFIGS:
    print(f'\n{"="*60}')
    print(f'ABLATION: {run_name}')
    print(f'{"="*60}')

    abl_cfg = copy.deepcopy(cfg)
    abl_cfg['experiment']['backbone'] = ABLATION_BACKBONE
    abl_cfg['experiment']['dataset']  = ABLATION_DATASET
    abl_cfg['experiment']['name']     = run_name
    abl_cfg['training']['epochs']     = ABLATION_EPOCHS
    abl_cfg['quantize'].update({
        'use_sparse':  use_sparse,
        'use_pot_bn':  use_pot_bn,
        'use_pot_act': use_pot_act,
        'use_ewgs':    use_ewgs,
    })

    res = run(abl_cfg)
    ablation_results.append({
        'name':     run_name,
        'test_acc': res['test_acc'],
        'sparsity': res['final_sparsity'],
        'energy':   res['final_ops']['energy_GpJ'],
    })

print('\n=== Ablation Summary ===')
print(f"{'-'*70}")
print(f"{'Name':<30} {'TestAcc':>8} {'Sparsity':>10} {'Energy(GpJ)':>12}")
print(f"{'-'*70}")
BASELINE_ENERGY = 0.1887
for r in ablation_results:
    ratio = f"{BASELINE_ENERGY/r['energy']:.1f}×" if r['energy'] > 0 else "—"
    print(f"{r['name']:<30} {r['test_acc']:>8.4f} {r['sparsity']:>10.2%} "
          f"{r['energy']:>12.4f}  {ratio}")

---
## Results Summary

In [ ]:
!python scripts/summarize_results.py